In [1]:
# Import libries
import numpy as np
import pandas as pd
import os
import re
import glob
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
import pysam
# from scipy.stats import mannwhitneyu, stats
# from statannotations.Annotator import Annotator

current_directory = os.getcwd()
print("Current Directory:", current_directory)
pd.set_option("display.max_columns", None)


Current Directory: /mnt/NAS3/home/jiwon/ECTRES/python


In [2]:
manifest=pd.read_csv('../manifest/sample_mapping_20260507.csv')
manifest.head(2)
parental_id = manifest[manifest['sample_id']=='parental']['aliquot_barcode'].unique().tolist()
parental_id

['ECTRES-H2170-0001-TPX-A01-WGS-3YV111',
 'ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985',
 'ECTRES-EFM19-0001-TPX-A01-WGS-2PV977']

In [4]:
# ECTRES_clones_nf_dna_bam_oncoanalyser.csv

oncoanalyser=pd.read_csv('../manifest/ECTRES_clones_nf_dna_fastqs.csv')
oncoanalyser.head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,RGID,RGPL,RGPU,RGLB,RGDT,RGCN,FQ1,FQ2,action
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_1,XY,2333V.6,ILLUMINA,2333VCLT4.6,ZKDN250032992,NaN,CBM,/mnt/NAS3/home/mary/rawData/ECTRES/X209SC25116...,/mnt/NAS3/home/mary/rawData/ECTRES/X209SC25116...,run
1,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_1,XY,232NW.1,ILLUMINA,232NW2LT3.1,ZKDN250032992,NaN,CBM,/mnt/NAS3/home/mary/rawData/ECTRES/X209SC25116...,/mnt/NAS3/home/mary/rawData/ECTRES/X209SC25116...,run


In [5]:
oncoanalyser['source_barcode'].unique()

array(['ECGI1', 'H2170', 'EFM19'], dtype=object)

In [17]:
df = oncoanalyser[oncoanalyser['aliquot_barcode'].isin(parental_id)].copy()

df['Patient_Barcode'] = df['patient_barcode']
df['Sex'] = df['gender'].map({'XY': 'M', 'XX': 'F'})
cell_line_map = {
    'H2170': 'NCI-H2170',
    'EFM19': 'EFM-192A',
    'ECGI1': 'EC-GI-10'
}

# [Diagnosis 컬럼용 암종 풀네임]
diagnosis_map = {
    'H2170': 'Squamous Cell Lung Carcinoma',
    'EFM19': 'Breast Carcinoma',
    'ECGI1': 'Esophageal Squamous Cell Carcinoma'
}

# [Cancer_Type 컬럼용 TCGA 약어]
cancer_type_map = {
    'H2170': 'LUSC',
    'EFM19': 'BRCA',
    'ECGI1': 'ESCA'
}

# 데이터프레임에 각각 매핑하여 컬럼 생성 및 업데이트
df['cell_line'] = df['source_barcode'].map(cell_line_map)
df['Diagnosis'] = df['source_barcode'].map(cancer_type_map)


# 5. Project_barcode 컬럼 ('-' 기준 맨 앞 마디)
df['Project_barcode'] = df['patient_barcode'].str.split('-').str[0]
# 결과 확인
df.head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,RGID,RGPL,RGPU,RGLB,RGDT,RGCN,FQ1,FQ2,action,Patient_Barcode,Sex,cell_line,Diagnosis,Project_barcode
60,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NaN,XY,22CYN.1,ILLUMINA,22CYNGLT4.1,TN2411D2435,NaN,CBM,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,run,ECTRES-H2170-0001,M,NCI-H2170,LUSC,ECTRES
61,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,NaN,XY,22NJL.8,ILLUMINA,22NJLGLT4.8,DKDN250022574,NaN,CBM,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,run,ECTRES-ECGI1-0001,M,EC-GI-10,ESCA,ECTRES


In [20]:
df.columns

Index(['aliquot_barcode', 'source_barcode', 'sample_barcode',
       'patient_barcode', 'sample_type', 'tumor_or_normal', 'sequence_type',
       'sample_legacy_id', 'gender', 'RGID', 'RGPL', 'RGPU', 'RGLB', 'RGDT',
       'RGCN', 'FQ1', 'FQ2', 'action', 'Patient_Barcode', 'Sex', 'cell_line',
       'Diagnosis', 'Project_barcode'],
      dtype='object')

In [19]:
df1=df[['Patient_Barcode', 'Sex','Diagnosis','Project_barcode']]
df1['Notes']='cell_line'
df1.head()

<ipython-input-19-decd1ff6bd5e>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['Notes']='cell_line'


,Patient_Barcode,Sex,Diagnosis,Project_barcode,Notes
60,ECTRES-H2170-0001,M,LUSC,ECTRES,cell_line
61,ECTRES-ECGI1-0001,M,ESCA,ECTRES,cell_line
62,ECTRES-EFM19-0001,F,BRCA,ECTRES,cell_line


In [21]:
df['Sample_Barcode']=df['sample_barcode']
df['Tumor_or_Normal'] = df['tumor_or_normal'].map({'tumor': 'Tumor', 'normal': 'Normal'})
tissue_site_map = {
    'H2170': 'Lung',
    'EFM19': 'pleural_effusion',
    'ECGI1': 'lymph_node'
}
df['Tissue_Site'] = df['source_barcode'].map(tissue_site_map)

df2=df[['Sample_Barcode','Patient_Barcode','Tumor_or_Normal','Tissue_Site']]
df2['Notes']=''
df2.head()

<ipython-input-21-9e22b8aa9df7>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Notes']=''


,Sample_Barcode,Patient_Barcode,Tumor_or_Normal,Tissue_Site,Notes
60,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,Tumor,Lung,
61,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,Tumor,lymph_node,
62,ECTRES-EFM19-0001-TPX-A01,ECTRES-EFM19-0001,Tumor,pleural_effusion,


In [22]:
df['Tumor_Aliquot_Barcode'] = df['aliquot_barcode']
df['Normal_Aliquot_Barcode'] = ""
df['Tool_Name'] = 'ichorCNA'
df['Tool_Version'] = '0.3.2'
df['Reference_Genome'] = 'GRCh37'

df.head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,RGID,RGPL,RGPU,RGLB,RGDT,RGCN,FQ1,FQ2,action,Patient_Barcode,Sex,cell_line,Diagnosis,Project_barcode,Sample_Barcode,Tumor_or_Normal,Tissue_Site,Tumor_Aliquot_Barcode,Normal_Aliquot_Barcode,Tool_Name,Tool_Version,Reference_Genome
60,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NaN,XY,22CYN.1,ILLUMINA,22CYNGLT4.1,TN2411D2435,NaN,CBM,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,run,ECTRES-H2170-0001,M,NCI-H2170,LUSC,ECTRES,ECTRES-H2170-0001-TPX-A01,Tumor,Lung,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
61,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,NaN,XY,22NJL.8,ILLUMINA,22NJLGLT4.8,DKDN250022574,NaN,CBM,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,run,ECTRES-ECGI1-0001,M,EC-GI-10,ESCA,ECTRES,ECTRES-ECGI1-0001-TPX-A01,Tumor,lymph_node,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,,ichorCNA,0.3.2,GRCh37


In [25]:
df

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,RGID,RGPL,RGPU,RGLB,RGDT,RGCN,FQ1,FQ2,action,Patient_Barcode,Sex,cell_line,Diagnosis,Project_barcode,Sample_Barcode,Tumor_or_Normal,Tissue_Site,Tumor_Aliquot_Barcode,Normal_Aliquot_Barcode,Tool_Name,Tool_Version,Reference_Genome
60,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NaN,XY,22CYN.1,ILLUMINA,22CYNGLT4.1,TN2411D2435,NaN,CBM,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,run,ECTRES-H2170-0001,M,NCI-H2170,LUSC,ECTRES,ECTRES-H2170-0001-TPX-A01,Tumor,Lung,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
61,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,NaN,XY,22NJL.8,ILLUMINA,22NJLGLT4.8,DKDN250022574,NaN,CBM,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,run,ECTRES-ECGI1-0001,M,EC-GI-10,ESCA,ECTRES,ECTRES-ECGI1-0001-TPX-A01,Tumor,lymph_node,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,,ichorCNA,0.3.2,GRCh37
62,ECTRES-EFM19-0001-TPX-A01-WGS-2PV977,EFM19,ECTRES-EFM19-0001-TPX-A01,ECTRES-EFM19-0001,TP,tumor,WGS,NaN,XX,22NJL.7,ILLUMINA,22NJLGLT4.7,DKDN250022572,NaN,CBM,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,/mnt/NAS3/home/mary/rawData/DRUGBR/X201SC25055...,run,ECTRES-EFM19-0001,F,EFM-192A,BRCA,ECTRES,ECTRES-EFM19-0001-TPX-A01,Tumor,pleural_effusion,ECTRES-EFM19-0001-TPX-A01-WGS-2PV977,,ichorCNA,0.3.2,GRCh37


In [28]:
import os
import pandas as pd
from glob import glob

# 1. 절대 경로 지정
base_dir = "/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/results/ichorCNA/1MB/ploidy234/ichorCNA"

# 2. 파일 목록 및 절대 경로 변환
all_files = glob(os.path.join(base_dir, "**", "*"), recursive=True)
absolute_files = [os.path.abspath(f) for f in all_files]

path_df = pd.DataFrame({'File_Path': absolute_files})

# 3. 에러 유발하는 apply 대신, 리스트 컴프리헨션으로 직접 바코드 파싱
patient_barcodes = []
aliquot_barcodes = []

for path in path_df['File_Path']:
    rel_path = os.path.relpath(path, base_dir)
    parts = rel_path.split(os.sep)
    
    # [patient_barcode, aliquot_barcode] 구조가 나오는지 확인
    if len(parts) >= 2:
        patient_barcodes.append(parts[0])
        aliquot_barcodes.append(parts[1])
    else:
        patient_barcodes.append(None)
        aliquot_barcodes.append(None)

# 4. 데이터프레임에 각각 컬럼으로 직접 주입 (ValueError 완전 방지)
path_df['patient_barcode'] = patient_barcodes
path_df['aliquot_barcode'] = aliquot_barcodes

# 5. 불필요한 결측치 및 중복 행 정제
path_df = path_df.dropna(subset=['patient_barcode', 'aliquot_barcode']).drop_duplicates()

# 결과 확인
path_df.head()

,File_Path,patient_barcode,aliquot_barcode
3,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001,ECTRES-H2170-0001-TPX-A25-WGS-MSZHL2
4,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001,ECTRES-H2170-0001-TPX-A29-WGS-HQRCV3
5,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001,ECTRES-H2170-0001-TPX-A17-WGS-JXAF26
6,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001,ECTRES-H2170-0001-TPX-A30-WGS-KUB89Z
7,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001,ECTRES-H2170-0001-TPX-A23-WGS-NZH3KA


In [29]:
path_df.shape

(2926, 3)

In [30]:
df.columns

Index(['aliquot_barcode', 'source_barcode', 'sample_barcode',
       'patient_barcode', 'sample_type', 'tumor_or_normal', 'sequence_type',
       'sample_legacy_id', 'gender', 'RGID', 'RGPL', 'RGPU', 'RGLB', 'RGDT',
       'RGCN', 'FQ1', 'FQ2', 'action', 'Patient_Barcode', 'Sex', 'cell_line',
       'Diagnosis', 'Project_barcode', 'Sample_Barcode', 'Tumor_or_Normal',
       'Tissue_Site', 'Tumor_Aliquot_Barcode', 'Normal_Aliquot_Barcode',
       'Tool_Name', 'Tool_Version', 'Reference_Genome'],
      dtype='object')

In [31]:
df_df = pd.merge(df, path_df, on=['patient_barcode', 'aliquot_barcode'], how='inner')

# 2. 앞서 정해둔 나머지 컬럼들 한 번에 채워넣기
df_df['Tumor_Aliquot_Barcode'] = df_df['aliquot_barcode']
df_df['Normal_Aliquot_Barcode'] = ""  # Normal은 비우기
df_df['Tool_Name'] = 'ichorCNA'
df_df['Tool_Version'] = '0.3.2'
df_df['Reference_Genome'] = 'GRCh37'

df_df.columns

Index(['aliquot_barcode', 'source_barcode', 'sample_barcode',
       'patient_barcode', 'sample_type', 'tumor_or_normal', 'sequence_type',
       'sample_legacy_id', 'gender', 'RGID', 'RGPL', 'RGPU', 'RGLB', 'RGDT',
       'RGCN', 'FQ1', 'FQ2', 'action', 'Patient_Barcode', 'Sex', 'cell_line',
       'Diagnosis', 'Project_barcode', 'Sample_Barcode', 'Tumor_or_Normal',
       'Tissue_Site', 'Tumor_Aliquot_Barcode', 'Normal_Aliquot_Barcode',
       'Tool_Name', 'Tool_Version', 'Reference_Genome', 'File_Path'],
      dtype='object')

In [32]:
df3=df_df[['File_Path','Tumor_Aliquot_Barcode', 'Normal_Aliquot_Barcode',
       'Tool_Name', 'Tool_Version', 'Reference_Genome']]
df3.head()

,File_Path,Tumor_Aliquot_Barcode,Normal_Aliquot_Barcode,Tool_Name,Tool_Version,Reference_Genome
0,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
1,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
2,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
3,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37
4,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37


In [38]:
import os
import pandas as pd

def map_ichorcna_files(path):
    filename = os.path.basename(path)
    
    # 1. 텍스트 및 세그먼트 데이터 파일 규칙 매칭
    if filename.endswith('.seg.txt'):
        return pd.Series(['SEG', '.seg.txt', 'Copy Number Segment', 'ichorCNA copy number segments'])
    elif filename.endswith('.params.txt'):
        return pd.Series(['TXT', '.params.txt', 'Purity/Ploidy Summary', 'ichorCNA tumor fraction estimates'])
    elif filename.endswith('.cna.seg'):
        return pd.Series(['SEG', '.cna.seg', 'Copy Number Segment', 'ichorCNA cna segment file'])
    elif filename.endswith('.correctedDepth.txt'):
        return pd.Series(['TXT', '.correctedDepth.txt', 'Read Depth Ratio', 'ichorCNA GC-corrected and normalized read depth'])
    
    # 2. PDF 시각화 파일 규칙 매칭 (tree 결과에 나온 유형별 분류)
    elif filename.endswith('.pdf'):
        if '_genomeWide_all_sols.pdf' in filename:
            return pd.Series(['PDF', '_genomeWide_all_sols.pdf', 'Copy Number Plot', 'ichorCNA genome-wide plots for all parameter solutions'])
        elif '_genomeWide.pdf' in filename:
            return pd.Series(['PDF', '_genomeWide.pdf', 'Copy Number Plot', 'ichorCNA optimal genome-wide copy number plot'])
        elif '_genomeWide_n0-p' in filename:
            # p1, p2, p3, p4 등 특정 플로이디 솔루션 플롯 처리
            p_num = filename.split('_n0-')[-1].replace('.pdf', '')
            return pd.Series(['PDF', f"_genomeWide_n0-{p_num}.pdf", 'Copy Number Plot', f"ichorCNA genome-wide plot for solution {p_num}"])
        elif '_CNA_chr' in filename:
            # chr1, chr2, chrX 등 염색체별 플롯 처리
            chr_name = filename.split('_CNA_')[-1].replace('.pdf', '')
            return pd.Series(['PDF', f"_CNA_{chr_name}.pdf", 'Copy Number Plot', f"ichorCNA chromosome-level copy number plot for {chr_name}"])
        elif '_bias.pdf' in filename:
            return pd.Series(['PDF', '_bias.pdf', 'QC Plot', 'ichorCNA GC and mappability bias diagnostic plot'])
        elif '_correct.pdf' in filename:
            return pd.Series(['PDF', '_correct.pdf', 'QC Plot', 'ichorCNA copy number correction diagnostic plot'])
        elif '_tpdf.pdf' in filename:
            return pd.Series(['PDF', '_tpdf.pdf', 'QC Plot', 'ichorCNA student-t distribution parameter plot'])
        else:
            return pd.Series(['PDF', '.pdf', 'Plot', 'ichorCNA supplementary plot'])
            
    # 3. 디렉토리 자체이거나 매칭되지 않는 임시 파일 예외처리
    else:
        return pd.Series(['UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN'])

# final_df에 4개 컬럼 동시 생성 및 매핑 데이터 적용
df_df[['File_Format', 'File_Suffix', 'Data_Type', 'Description']] = df_df['File_Path'].apply(map_ichorcna_files)

# 상위 폴더 경로 등 불필요한 UNKNOWN 행이 섞여있다면 깔끔하게 드롭
df_df = df_df[df_df['File_Format'] != 'UNKNOWN']

# 주피터 노트북 출력으로 잘 드려맞았는지 샘플 확인
df_df[['File_Path', 'File_Format', 'File_Suffix', 'Data_Type', 'Description']].head(15)

,File_Path,File_Format,File_Suffix,Data_Type,Description
1,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,TXT,.params.txt,Purity/Ploidy Summary,ichorCNA tumor fraction estimates
2,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,SEG,.seg.txt,Copy Number Segment,ichorCNA copy number segments
3,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,SEG,.cna.seg,Copy Number Segment,ichorCNA cna segment file
4,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,TXT,.correctedDepth.txt,Read Depth Ratio,ichorCNA GC-corrected and normalized read depth
6,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide_n0-p1.pdf,Copy Number Plot,ichorCNA genome-wide plot for solution p1
7,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide_n0-p2.pdf,Copy Number Plot,ichorCNA genome-wide plot for solution p2
8,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide_n0-p3.pdf,Copy Number Plot,ichorCNA genome-wide plot for solution p3
9,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide_n0-p4.pdf,Copy Number Plot,ichorCNA genome-wide plot for solution p4
10,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide_all_sols.pdf,Copy Number Plot,ichorCNA genome-wide plots for all parameter s...
11,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,PDF,_genomeWide.pdf,Copy Number Plot,ichorCNA optimal genome-wide copy number plot


In [41]:
final_df=df_df.copy()
# 1. 맨 앞으로 보낼 핵심 메타데이터 컬럼 순서 정의
front_columns = [
    'Tool_Name', 
    'Tool_Version', 
    'File_Format', 
    'File_Suffix', 
    'Data_Type', 
    'Description'
]

remaining_columns = [col for col in final_df.columns if col not in front_columns]
final_cols_order = front_columns + remaining_columns

final_df = final_df.reset_index(drop=True)
final_df.head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,RGID,RGPL,RGPU,RGLB,RGDT,RGCN,FQ1,FQ2,action,Patient_Barcode,Sex,cell_line,Diagnosis,Project_barcode,Sample_Barcode,Tumor_or_Normal,Tissue_Site,Tumor_Aliquot_Barcode,Normal_Aliquot_Barcode,Tool_Name,Tool_Version,Reference_Genome,File_Path,File_Format,File_Suffix,Data_Type,Description
0,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NaN,XY,22CYN.1,ILLUMINA,22CYNGLT4.1,TN2411D2435,NaN,CBM,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,run,ECTRES-H2170-0001,M,NCI-H2170,LUSC,ECTRES,ECTRES-H2170-0001-TPX-A01,Tumor,Lung,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,TXT,.params.txt,Purity/Ploidy Summary,ichorCNA tumor fraction estimates
1,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NaN,XY,22CYN.1,ILLUMINA,22CYNGLT4.1,TN2411D2435,NaN,CBM,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,/mnt/NAS2/home/mary/rawData/TBD241712_22968_20...,run,ECTRES-H2170-0001,M,NCI-H2170,LUSC,ECTRES,ECTRES-H2170-0001-TPX-A01,Tumor,Lung,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,,ichorCNA,0.3.2,GRCh37,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECRES/resul...,SEG,.seg.txt,Copy Number Segment,ichorCNA copy number segments


In [42]:
df4=final_df[front_columns].drop_duplicates()

print(df4.shape)
df4.head(2)

(36, 6)


,Tool_Name,Tool_Version,File_Format,File_Suffix,Data_Type,Description
0,ichorCNA,0.3.2,TXT,.params.txt,Purity/Ploidy Summary,ichorCNA tumor fraction estimates
1,ichorCNA,0.3.2,SEG,.seg.txt,Copy Number Segment,ichorCNA copy number segments


In [44]:
save_dir='/mnt/NAS3/home/jiwon/database/'

df1.to_csv(f'{save_dir}01_patients.csv', index=False)
df2.to_csv(f'{save_dir}02_samples.csv', index=False)
df3.to_csv(f'{save_dir}03_files.csv', index=False)
df4.to_csv(f'{save_dir}04_tool_output_types.csv', index=False)


## mosdepth

In [4]:
import pandas as pd
import random
import string
import os

def generate_file_id():
    chars = string.ascii_lowercase + string.digits
    return "-".join("".join(random.choices(chars, k=5)) for _ in range(4))

# 1. manifest 읽기
manifest = pd.read_csv("/mnt/NAS3/home/jiwon/BIOCHP/manifest/BIOCHP_JW_WGS_manifest_fqs.csv")

# 2. example_files.csv 컬럼
columns = [
    "file_id", "file_name", "file_format", "file_size_gb", "file_md5",
    "file_path", "created_date", "tumor_aliquot_barcode", "normal_aliquot_barcode",
    "analysis_type", "tool_name", "tool_version", "reference_genome",
    "experimental_strategy", "data_category", "data_type"
]

rows = []
for _, r in manifest.iterrows():
    if str(r["tumor_or_normal"]).lower() == "tumor":
        tumor_barcode, normal_barcode = r["aliquot_barcode"], ""
    else:
        tumor_barcode, normal_barcode = "", r["aliquot_barcode"]

    analysis_type = "Tumor-Only" if normal_barcode == "" else "Tumor-Normal"

    for fq_col, md5_col in [("FQ1", "md5sum.fq1"), ("FQ2", "md5sum.fq2")]:
        rows.append({
            "file_id": generate_file_id(),
            "file_name": os.path.basename(r[fq_col]),
            "file_format": "FASTQ",
            "file_size_gb": "",  # TotalBases(Gb)는 용량이 아니라 염기수 -> 아래 설명 참고
            "file_md5": r[md5_col],
            "file_path": r[fq_col],
            "created_date": r["sample_date"],
            "tumor_aliquot_barcode": tumor_barcode,
            "normal_aliquot_barcode": normal_barcode,
            "analysis_type": analysis_type,
            "tool_name": "",
            "tool_version": "",
            "reference_genome": "",
            "experimental_strategy": r["sequence_type"],
            "data_category": "Sequencing Reads",
            "data_type": "Raw Reads",
        })

fastq_df = pd.DataFrame(rows, columns=columns)
fastq_df.head(10)

,file_id,file_name,file_format,file_size_gb,file_md5,file_path,created_date,tumor_aliquot_barcode,normal_aliquot_barcode,analysis_type,tool_name,tool_version,reference_genome,experimental_strategy,data_category,data_type
0,80lmv-f30bm-s8vw7-e8hjh,Colo320HSR_1.fq.gz,FASTQ,,607f01b62eddcb6d8a47d8da30cfec8d,/mnt/NAS4/home/mary/SKKURT/rawData/TBD240086_1...,20240219,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
1,2bnh2-ieg8t-im1bk-op9i4,Colo320HSR_2.fq.gz,FASTQ,,eef32ec3eb64ebef0517c632f5da5d18,/mnt/NAS4/home/mary/SKKURT/rawData/TBD240086_1...,20240219,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
2,8cewg-dungg-fs98k-bbgpc,Colo320DM_1.fq.gz,FASTQ,,9b68b9eb555e07544c4b64c73d6bfe36,/mnt/NAS4/home/mary/SKKURT/rawData/TBD240086_1...,20240219,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
3,0l0j4-t5xm7-53pql-h29uy,Colo320DM_2.fq.gz,FASTQ,,ee962b490984f8bd82cbb1fcacb0c2eb,/mnt/NAS4/home/mary/SKKURT/rawData/TBD240086_1...,20240219,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
4,hf133-zokec-m950m-xhv3x,2__collagen_colo320DM_1.fq.gz,FASTQ,,d2fe80b71e41094e045d6f5332769203,/mnt/NAS/storage/SKKU_MED_biochip/TBD241134_21...,20240909,BIOCHP-SKKUM-0001-TPX-A01-WGS-UVH5PA,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
5,o3rf2-3vcyt-h800j-f3qme,2__collagen_colo320DM_2.fq.gz,FASTQ,,d42b486571f800df2399f96bac88cb28,/mnt/NAS/storage/SKKU_MED_biochip/TBD241134_21...,20240909,BIOCHP-SKKUM-0001-TPX-A01-WGS-UVH5PA,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
6,pkmsj-ugr6v-ldhbn-g2bw6,2__collagen_colo320HSR_1.fq.gz,FASTQ,,c8e22ed6d4cbc8bb1cb1f9c410e48096,/mnt/NAS/storage/SKKU_MED_biochip/TBD241134_21...,20240909,BIOCHP-SKKUM-0002-TPX-A01-WGS-SCZHEV,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
7,8r9ph-ro59h-k6cgp-qzypm,2__collagen_colo320HSR_2.fq.gz,FASTQ,,29c1750b6214e1dfb1d297cc37641966,/mnt/NAS/storage/SKKU_MED_biochip/TBD241134_21...,20240909,BIOCHP-SKKUM-0002-TPX-A01-WGS-SCZHEV,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
8,bna8c-vtt7s-7n5x6-pps23,DM_1_1.fq.gz,FASTQ,,598b0e4a06e90bd66064503749d6f107,/mnt/NAS/storage/SKKU_MED_biochip/TBD241935_23...,20250210,BIOCHP-SKKUM-0001-TPX-A02-WGS-CGENDZ,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads
9,1xc7z-g40w7-8cwog-3hytm,DM_1_2.fq.gz,FASTQ,,c226e204ab7f095da50d6d60d33cc853,/mnt/NAS/storage/SKKU_MED_biochip/TBD241935_23...,20250210,BIOCHP-SKKUM-0001-TPX-A02-WGS-CGENDZ,,Tumor-Only,,,,WGS,Sequencing Reads,Raw Reads


In [5]:
import os

# ==== 서버상 mosdepth 결과 폴더 경로 입력 ====
MOSDEPTH_BASE_DIR = "/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam"
MOSDEPTH_SUBDIR = "covWgs_mosdepth"

# ==== tool_version 입력 ====
# align_dna.sif 사용 시 "v0.3.3", 매뉴얼 실행 시 singularity 이미지 버전 확인 후 입력
MOSDEPTH_TOOL_VERSION = "v0.3.3"  # <- 실제 버전으로 수정

# mosdepth가 생성하는 파일 종류 (example_files.csv 기준)
MOSDEPTH_SUFFIXES = [
    ("mosdepth.global.dist.txt", "TXT", "Coverage Distribution"),
    ("mosdepth.region.dist.txt", "TXT", "Per-region Coverage Distribution"),
    ("mosdepth.summary.txt",     "TXT", "Coverage Summary"),
    ("per-base.bed.gz",          "BED", "Per-base Coverage"),
    ("per-base.bed.gz.csi",      "CSI", ""),
    ("regions.bed.gz",           "BED", "Per-region Coverage"),
    ("regions.bed.gz.csi",       "CSI", ""),
]

rows = []
for _, r in manifest.iterrows():
    barcode = r["aliquot_barcode"]
    if str(r["tumor_or_normal"]).lower() == "tumor":
        tumor_barcode, normal_barcode = barcode, ""
    else:
        tumor_barcode, normal_barcode = "", barcode

    analysis_type = "Tumor-Only" if normal_barcode == "" else "Tumor-Normal"

    for suffix, fmt, data_type in MOSDEPTH_SUFFIXES:
        file_name = f"{barcode}.{suffix}"
        rows.append({
            "file_id": generate_file_id(),
            "file_name": file_name,
            "file_format": fmt,
            "file_size_gb": "",
            "file_md5": "",
            "file_path": os.path.join(MOSDEPTH_BASE_DIR, barcode, MOSDEPTH_SUBDIR, file_name),
            "created_date": "",
            "tumor_aliquot_barcode": tumor_barcode,
            "normal_aliquot_barcode": normal_barcode,
            "analysis_type": analysis_type,
            "tool_name": "mosdepth",
            "tool_version": MOSDEPTH_TOOL_VERSION,
            "reference_genome": "",
            "experimental_strategy": r["sequence_type"],
            "data_category": "",
            "data_type": data_type,
        })

mosdepth_df = pd.DataFrame(rows, columns=columns)
mosdepth_df.head(10)

,file_id,file_name,file_format,file_size_gb,file_md5,file_path,created_date,tumor_aliquot_barcode,normal_aliquot_barcode,analysis_type,tool_name,tool_version,reference_genome,experimental_strategy,data_category,data_type
0,rdles-1nv93-glqjl-zx6lb,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Distribution
1,2dxdi-4az4d-m6nts-63hk6,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage Distribution
2,2jpm7-b686m-tfs7x-yp4kn,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Summary
3,fp9q8-g6qea-vftxb-11w3u,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.per-base....,BED,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-base Coverage
4,9cd5x-htmza-gv904-23muk,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.per-base....,CSI,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,
5,4sfar-qapq0-hxwq9-ein3i,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.regions.b...,BED,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage
6,ghv2f-7aqnf-hq6ct-s8ifb,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.regions.b...,CSI,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,
7,ikzsd-toozd-jgjey-g2lm3,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Distribution
8,ma4so-v7ui5-e7m9d-qb5sq,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage Distribution
9,g4cox-0v3xh-2kre8-dhvj9,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361.mosdepth....,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Summary


In [12]:
MOSDEPTH_BASE_DIR = "/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam"
MOSDEPTH_SUBDIR = "covWgs_mosdepth"
MOSDEPTH_TOOL_VERSION = "v0.3.3"

# 실제 서버 파일명 규칙에 맞게 수정: -WGSCov 접미사, per-base 없이 5개만
MOSDEPTH_SUFFIXES = [
    ("WGSCov.mosdepth.global.dist.txt", "TXT", "Coverage Distribution"),
    ("WGSCov.mosdepth.region.dist.txt", "TXT", "Per-region Coverage Distribution"),
    ("WGSCov.mosdepth.summary.txt",     "TXT", "Coverage Summary"),
    ("WGSCov.regions.bed.gz",           "BED", "Per-region Coverage"),
    ("WGSCov.regions.bed.gz.csi",       "CSI", ""),
]

rows = []
for _, r in manifest.iterrows():
    barcode = r["aliquot_barcode"]
    if str(r["tumor_or_normal"]).lower() == "tumor":
        tumor_barcode, normal_barcode = barcode, ""
    else:
        tumor_barcode, normal_barcode = "", barcode
    analysis_type = "Tumor-Only" if normal_barcode == "" else "Tumor-Normal"

    for suffix, fmt, data_type in MOSDEPTH_SUFFIXES:
        file_name = f"{barcode}-{suffix}"   # 점(.)이 아니라 대시(-)로 연결
        rows.append({
            "file_id": generate_file_id(),
            "file_name": file_name,
            "file_format": fmt,
            "file_size_gb": "",
            "file_md5": "",
            "file_path": os.path.join(MOSDEPTH_BASE_DIR, barcode, MOSDEPTH_SUBDIR, file_name),
            "created_date": "",
            "tumor_aliquot_barcode": tumor_barcode,
            "normal_aliquot_barcode": normal_barcode,
            "analysis_type": analysis_type,
            "tool_name": "mosdepth",
            "tool_version": MOSDEPTH_TOOL_VERSION,
            "reference_genome": "",
            "experimental_strategy": r["sequence_type"],
            "data_category": "",
            "data_type": data_type,
        })

mosdepth_df = pd.DataFrame(rows, columns=columns)
mosdepth_df.head(10)

,file_id,file_name,file_format,file_size_gb,file_md5,file_path,created_date,tumor_aliquot_barcode,normal_aliquot_barcode,analysis_type,tool_name,tool_version,reference_genome,experimental_strategy,data_category,data_type
0,ptbsz-1ixeb-8tp68-ixz7v,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Distribution
1,jtfn6-jvy7q-88ovz-egzni,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage Distribution
2,1df7f-5p532-0f3ue-irxx1,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Summary
3,xmp6i-vyvcu-stm6j-gcps0,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361-WGSCov.re...,BED,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage
4,hmw6q-wsrlo-2iic4-qlhyj,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361-WGSCov.re...,CSI,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,
5,jipa8-d88n3-m9rn2-s1ja0,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Distribution
6,6jefa-6c4yr-covbu-a86xs,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage Distribution
7,mojkr-onnud-57u00-uq5gm,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Coverage Summary
8,5usqf-noevf-ziiqp-130q4,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361-WGSCov.re...,BED,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,Per-region Coverage
9,3xujn-3ojnh-uribe-klur4,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361-WGSCov.re...,CSI,,,/mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/res...,,BIOCHP-SKKUM-0001-TP0-A01-WGS-2OJ361,,Tumor-Only,mosdepth,v0.3.3,,WGS,,


In [13]:
BIOCHP_files_df = pd.concat([fastq_df, mosdepth_df], ignore_index=True)
# BIOCHP_files_df.to_csv("(initial)_files.csv", index=False)

In [8]:
import pandas as pd
import os

bam_manifest = pd.read_csv("/mnt/NAS3/home/jiwon/ECTRES/manifest/ECTRES_clones_nf_dna_bam_update.csv")

columns = [
    "file_id", "file_name", "file_format", "file_size_gb", "file_md5",
    "file_path", "created_date", "tumor_aliquot_barcode", "normal_aliquot_barcode",
    "analysis_type", "tool_name", "tool_version", "reference_genome",
    "experimental_strategy", "data_category", "data_type"
]

ALIGN_TOOL_VERSION = "v0.3.3"  # 경로에 align_dna 있어서 확정, 다르면 수정
REFERENCE_GENOME = "GRCh37"    # 경로에서 확인됨

# 1. aligned reads (BAM/BAI) 행
aligned_rows = []
for _, r in bam_manifest.iterrows():
    barcode = r["aliquot_barcode"]
    if str(r["tumor_or_normal"]).lower() == "tumor":
        tumor_barcode, normal_barcode = barcode, ""
    else:
        tumor_barcode, normal_barcode = "", barcode
    analysis_type = "Tumor-Only" if normal_barcode == "" else "Tumor-Normal"

    for file_col, fmt, data_type in [("bam", "BAM", "Aligned Reads"), ("bai", "BAI", "")]:
        aligned_rows.append({
            "file_id": generate_file_id(),
            "file_name": os.path.basename(r[file_col]),
            "file_format": fmt,
            "file_size_gb": "",
            "file_md5": "",
            "file_path": r[file_col],
            "created_date": "",
            "tumor_aliquot_barcode": tumor_barcode,
            "normal_aliquot_barcode": normal_barcode,
            "analysis_type": analysis_type,
            "tool_name": "align_dna",
            "tool_version": ALIGN_TOOL_VERSION,
            "reference_genome": REFERENCE_GENOME,
            "experimental_strategy": r["sequence_type"],
            "data_category": "Sequencing Reads",
            "data_type": data_type,
        })

# 2. mosdepth 행 (이 manifest는 aliquot당 1행이라 dedup 불필요)
MOSDEPTH_BASE_DIR = "/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/results/qc_dna_bam"
MOSDEPTH_SUBDIR = "covWgs_mosdepth"

MOSDEPTH_SUFFIXES = [
    ("WGSCov.mosdepth.global.dist.txt", "TXT", "Coverage Distribution"),
    ("WGSCov.mosdepth.region.dist.txt", "TXT", "Per-region Coverage Distribution"),
    ("WGSCov.mosdepth.summary.txt",     "TXT", "Coverage Summary"),
    ("WGSCov.regions.bed.gz",           "BED", "Per-region Coverage"),
    ("WGSCov.regions.bed.gz.csi",       "CSI", ""),
]

mosdepth_rows = []
for _, r in bam_manifest.iterrows():
    barcode = r["aliquot_barcode"]
    if str(r["tumor_or_normal"]).lower() == "tumor":
        tumor_barcode, normal_barcode = barcode, ""
    else:
        tumor_barcode, normal_barcode = "", barcode
    analysis_type = "Tumor-Only" if normal_barcode == "" else "Tumor-Normal"

    for suffix, fmt, data_type in MOSDEPTH_SUFFIXES:
        file_name = f"{barcode}-{suffix}"
        mosdepth_rows.append({
            "file_id": generate_file_id(),
            "file_name": file_name,
            "file_format": fmt,
            "file_size_gb": "",
            "file_md5": "",
            "file_path": os.path.join(MOSDEPTH_BASE_DIR, barcode, MOSDEPTH_SUBDIR, file_name),
            "created_date": "",
            "tumor_aliquot_barcode": tumor_barcode,
            "normal_aliquot_barcode": normal_barcode,
            "analysis_type": analysis_type,
            "tool_name": "mosdepth",
            "tool_version": ALIGN_TOOL_VERSION,
            "reference_genome": REFERENCE_GENOME,
            "experimental_strategy": r["sequence_type"],
            "data_category": "",
            "data_type": data_type,
        })

# 3. 합치기
ECTRES_files_df = pd.DataFrame(aligned_rows + mosdepth_rows, columns=columns)
ECTRES_files_df.head(10)

,file_id,file_name,file_format,file_size_gb,file_md5,file_path,created_date,tumor_aliquot_barcode,normal_aliquot_barcode,analysis_type,tool_name,tool_version,reference_genome,experimental_strategy,data_category,data_type
0,knr0e-ij9oz-nbq64-o0oj3,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349.realn.mdu...,BAM,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,Aligned Reads
1,nfxk9-5t8s9-csl1x-4v3jr,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349.realn.mdu...,BAI,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,
2,zgwo9-34p8s-gvwdy-c3eg5,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949.realn.mdu...,BAM,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,Aligned Reads
3,dt4ze-knaql-ys1su-yj3xc,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949.realn.mdu...,BAI,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,
4,2o0l1-ooair-zqvxv-2gjak,ECTRES-ECGI1-0001-TPX-A11-WGS-9HJ669.realn.mdu...,BAM,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A11-WGS-9HJ669,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,Aligned Reads
5,i5ius-mmboe-nr4id-fxt3f,ECTRES-ECGI1-0001-TPX-A11-WGS-9HJ669.realn.mdu...,BAI,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A11-WGS-9HJ669,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,
6,nuilg-k8c08-k3sy9-z3tkf,ECTRES-ECGI1-0001-TPX-A12-WGS-4SL389.realn.mdu...,BAM,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A12-WGS-4SL389,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,Aligned Reads
7,m1yh0-8ai2s-zx11m-oh4ju,ECTRES-ECGI1-0001-TPX-A12-WGS-4SL389.realn.mdu...,BAI,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A12-WGS-4SL389,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,
8,bghd7-qmuqx-hist3-dl2an,ECTRES-ECGI1-0001-TPX-A13-WGS-3VZ640.realn.mdu...,BAM,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A13-WGS-3VZ640,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,Aligned Reads
9,geiph-26tes-dwi4j-we29b,ECTRES-ECGI1-0001-TPX-A13-WGS-3VZ640.realn.mdu...,BAI,,,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,,ECTRES-ECGI1-0001-TPX-A13-WGS-3VZ640,,Tumor-Only,align_dna,v0.3.3,GRCh37,WGS,Sequencing Reads,


In [9]:
ECTRES_files_df.tail()

,file_id,file_name,file_format,file_size_gb,file_md5,file_path,created_date,tumor_aliquot_barcode,normal_aliquot_barcode,analysis_type,tool_name,tool_version,reference_genome,experimental_strategy,data_category,data_type
534,f9kuv-87thm-z7ws4-8igep,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS,,Tumor-Only,mosdepth,v0.3.3,GRCh37,WGS,,Coverage Distribution
535,lny6b-rxz82-6d1an-0926i,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS,,Tumor-Only,mosdepth,v0.3.3,GRCh37,WGS,,Per-region Coverage Distribution
536,noiyy-3aaxs-wltm9-5hlae,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS-WGSCov.mo...,TXT,,,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS,,Tumor-Only,mosdepth,v0.3.3,GRCh37,WGS,,Coverage Summary
537,ifd0x-29t1i-09ae2-2jq0i,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS-WGSCov.re...,BED,,,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS,,Tumor-Only,mosdepth,v0.3.3,GRCh37,WGS,,Per-region Coverage
538,tah6w-nyhl5-4ttx0-43qb3,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS-WGSCov.re...,CSI,,,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,,ECTRES-H2170-0001-TPX-A31-WGS-VZD4GS,,Tumor-Only,mosdepth,v0.3.3,GRCh37,WGS,,


In [10]:
save_dir='/mnt/NAS3/home/jiwon/database/'

# BIOCHP: fastq_df + mosdepth_df 합치기 (이미 BIOCHP_files_df 만들어두셨으면 이 줄은 생략)
# BIOCHP_files_df = pd.concat([fastq_df, mosdepth_df], ignore_index=True)

# 최종 합치기 + 저장
SJW_files_df = pd.concat([BIOCHP_files_df, ECTRES_files_df], ignore_index=True)
SJW_files_df.to_csv(f"{save_dir}SJW_files.csv", index=False)

print(len(SJW_files_df), "행 저장 완료 -> SJW_files.csv")





611 행 저장 완료 -> SJW_files.csv


In [11]:
import os
import hashlib
from datetime import datetime

def compute_md5(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

missing_paths = []

for idx, row in SJW_files_df.iterrows():
    path = row["file_path"]

    if not os.path.exists(path):
        missing_paths.append(path)
        continue

    stat = os.stat(path)

    # 비어있는 값만 채움 (이미 값 있으면 덮어쓰지 않음)
    if not row["file_size_gb"]:
        SJW_files_df.at[idx, "file_size_gb"] = round(stat.st_size / (1024 ** 3), 4)

    if not row["created_date"]:
        SJW_files_df.at[idx, "created_date"] = datetime.fromtimestamp(stat.st_mtime).strftime("%Y%m%d")

    if not row["file_md5"]:
        SJW_files_df.at[idx, "file_md5"] = compute_md5(path)

# reference_genome 빈 값 -> GRCh37
SJW_files_df.loc[SJW_files_df["reference_genome"] == "", "reference_genome"] = "GRCh37"

print(f"경로에서 못 찾은 파일 수: {len(missing_paths)}")
for p in missing_paths[:20]:
    print(" -", p)

SJW_files_df.to_csv(f"{save_dir}SJW_files_update.csv", index=False)

경로에서 못 찾은 파일 수: 56
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361/covWgs_mosdepth/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth.global.dist.txt
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361/covWgs_mosdepth/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth.region.dist.txt
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361/covWgs_mosdepth/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.mosdepth.summary.txt
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361/covWgs_mosdepth/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.per-base.bed.gz
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361/covWgs_mosdepth/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ361.per-base.bed.gz.csi
 - /mnt/NAS3/home/jiwon/HL-GBM/scratch/BIOCHP/results/qc_dna_bam/BIOCHP-SKKUM-0002-TP0-A01-WGS-2OJ3

In [14]:
# 1. 기존 SJW_files_df에서 BIOCHP 관련 행 전부 제거
is_biochp = (
    SJW_files_df["tumor_aliquot_barcode"].astype(str).str.startswith("BIOCHP") |
    SJW_files_df["normal_aliquot_barcode"].astype(str).str.startswith("BIOCHP")
)
SJW_files_df = SJW_files_df[~is_biochp].reset_index(drop=True)

# 2. 새로 만든 BIOCHP_files_df(fastq_df + 고친 mosdepth_df) 통째로 붙이기
SJW_files_df = pd.concat([SJW_files_df, BIOCHP_files_df], ignore_index=True)

# 3. 빈 값만 채우기 (ECTRES는 이미 채워져 있어서 스킵, BIOCHP만 새로 계산됨)
missing_paths = []
for idx, row in SJW_files_df.iterrows():
    path = row["file_path"]
    if not os.path.exists(path):
        missing_paths.append(path)
        continue
    stat = os.stat(path)
    if not row["file_size_gb"]:
        SJW_files_df.at[idx, "file_size_gb"] = round(stat.st_size / (1024 ** 3), 4)
    if not row["created_date"]:
        SJW_files_df.at[idx, "created_date"] = datetime.fromtimestamp(stat.st_mtime).strftime("%Y%m%d")
    if not row["file_md5"]:
        SJW_files_df.at[idx, "file_md5"] = compute_md5(path)

SJW_files_df.loc[SJW_files_df["reference_genome"] == "", "reference_genome"] = "GRCh37"

print(f"경로에서 못 찾은 파일 수: {len(missing_paths)}")
for p in missing_paths[:20]:
    print(" -", p)

SJW_files_df.to_csv(f"{save_dir}SJW_files_update2.csv", index=False)

경로에서 못 찾은 파일 수: 0
